In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import xarray as xr
import hvplot.xarray
import numpy as np
import xdatasets as xd

import xhydro as xh
import xhydro.frequency_analysis as xhfa
import scipy.stats as stats
import pandas as pd

In [3]:
ds = xd.Query(
    **{
        "datasets":{
            "deh":{
                "id" :["023422*"],
                "regulated":["Natural"],
                "variables":["streamflow"],
            }
        }, "time":{"start": "1970-01-01", 
                   "minimum_duration":(15*365, 'd')},

  }
).data.squeeze().load()

# This dataset lacks some of the aforementioned attributes, so we need to add them.
ds["id"].attrs["cf_role"] = "timeseries_id"
ds["streamflow"].attrs = {"long_name": "Streamflow", "units": "m3 s-1", "standard_name": "water_volume_transport_in_river_channel", "cell_methods": "time: mean"}

ds

<xarray.Dataset>
Dimensions:        (time: 20454)
Coordinates: (12/15)
    drainage_area  float32 696.0
    end_date       datetime64[ns] 2023-11-19
    id             <U6 '023422'
    latitude       float32 46.17
    longitude      float32 -70.64
    name           object 'Famine'
    ...             ...
    spatial_agg    <U9 'watershed'
    start_date     datetime64[ns] 1970-01-01
  * time           (time) datetime64[ns] 1970-01-01 1970-01-02 ... 2025-12-31
    time_agg       <U4 'mean'
    timestep       <U1 'D'
    variable       <U10 'streamflow'
Data variables:
    streamflow     (time) float32 6.8 5.8 5.1 4.53 4.16 ... nan nan nan nan nan

In [4]:
# Here, we hide years with more than 15% of missing data.
ds_4fa = xh.indicators.get_yearly_op(ds, op="max", missing="pct", missing_options={"tolerance": 0.15})

ds_4fa

<xarray.Dataset>
Dimensions:                (time: 56)
Coordinates: (12/15)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    spatial_agg            <U9 'watershed'
    start_date             datetime64[ns] 1970-01-01
    time_agg               <U4 'mean'
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * time                   (time) datetime64[ns] 1970-01-01 ... 2025-01-01
Data variables:
    streamflow_max_annual  (time) float32 187.0 196.0 215.0 ... 158.3 nan nan
Attributes:
    cat:variable:          ('streamflow_max_annual',)
    cat:xrfreq:            AS-JAN
    cat:frequency:         yr
    cat:processing_level:  indicators
    cat:id:

In [5]:
ds_4fa.streamflow_max_annual.values

array([187. , 196. , 215. , 215. , 265. , 152. , 259. , 146. , 124. ,
       145. ,  91.4, 219. , 193. , 185. , 157. , 102. , 187. , 299. ,
       172. , 124. , 345. ,   nan,   nan, 162.6, 226. , 139.3, 182.3,
       124.6, 178.6, 101.2, 149.6, 175.9, 287.3, 111.8, 162.6, 180.3,
       262.9, 215.7, 286.1, 289.3, 211.7, 233. , 167.8, 192.9, 210.8,
       123.5, 184.6, 215.2, 288.8, 500.2, 138.3, 143.9, 199.4, 158.3,
         nan,   nan], dtype=float32)

In [6]:
# To speed up the Notebook, we'll only perform the analysis on a subset of variables
params = xhfa.local.fit(ds_4fa[["streamflow_max_annual"]], min_years=15, distributions=['genextreme'])
params

<xarray.Dataset>
Dimensions:                (dparams: 3, scipy_dist: 1)
Coordinates: (12/16)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    start_date             datetime64[ns] 1970-01-01
    time_agg               <U4 'mean'
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * dparams                (dparams) <U5 'c' 'loc' 'scale'
  * scipy_dist             (scipy_dist) <U10 'genextreme'
Data variables:
    streamflow_max_annual  (scipy_dist, dparams) float64 dask.array<chunksize=(1, 3), meta=np.ndarray>
Attributes:
    cat:variable:          ('streamflow_max_annual',)
    cat:xrfreq:            AS-JAN
    cat:frequency:         yr
    cat:processing_level:  indicators
    cat:id:

In [7]:
parameters = params.isel(scipy_dist=0)
parameters

<xarray.Dataset>
Dimensions:                (dparams: 3)
Coordinates: (12/16)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    start_date             datetime64[ns] 1970-01-01
    time_agg               <U4 'mean'
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * dparams                (dparams) <U5 'c' 'loc' 'scale'
    scipy_dist             <U10 'genextreme'
Data variables:
    streamflow_max_annual  (dparams) float64 dask.array<chunksize=(3,), meta=np.ndarray>
Attributes:
    cat:variable:          ('streamflow_max_annual',)
    cat:xrfreq:            AS-JAN
    cat:frequency:         yr
    cat:processing_level:  indicators
    cat:id:

In [9]:
shape = parameters.streamflow_max_annual.sel(dparams ='c').values
scale = parameters.streamflow_max_annual.sel(dparams ='scale').values
loc = parameters.streamflow_max_annual.sel(dparams ='loc').values

In [10]:
from scipy.stats import gamma, genextreme
dist = genextreme(shape, loc, scale)
variance = dist.var()

In [39]:
stats.genextreme.ppf([.99])

TypeError: _parse_args() missing 1 required positional argument: 'c'

In [34]:
T = [20, 50, 100, 200, 1000, 2000, 10000]
rp = xhfa.local.parametric_quantiles(params, t=T)
quantiles = rp.streamflow_max_annual.values


In [44]:
[1-(1/x)/2 for x in T]

[0.975, 0.99, 0.995, 0.9975, 0.9995, 0.99975, 0.99995]

In [45]:
# Valider le / 2
z = stats.norm.ppf([1-(1/x) for x in T])
up = [q + z*np.sqrt(variance) for q, z in zip(quantiles, z)]
down = [q - z*np.sqrt(variance) for q, z in zip(quantiles, z)]

In [46]:
df_analytic = pd.DataFrame({"0.05": down[0], "0.5": quantiles[0], "0.95": up[0]}, index=T)
df_analytic

,0.05,0.5,0.95
20,313.202696,326.978367,340.754038
50,370.867963,384.643634,398.419306
100,416.300684,430.076356,443.852027
200,463.552151,477.327822,491.103494
1000,581.101278,594.876949,608.652620
2000,635.383096,649.158767,662.934438
10000,770.694329,784.470000,798.245672


In [15]:
def calc_rvs(data, params, dist, nb_samples):

    data_length = len(data)

    params = params[~np.isnan(params)]
    samples = getattr(stats, dist).rvs(*params, size=(nb_samples, data_length))
    samples[:,np.isnan(data)] = np.nan
    return samples

In [16]:
# # Pour traiter plusieurs échantillon de tailles différentes
# mask = ~params.isnull().all(dim=['scipy_dist','dparams'])

# ds_data = ds_4fa[['streamflow_max_annual']].where(mask).dropna(dim='id', how='all')
# ds_data['params'] = params['streamflow_max_annual'].dropna(dim='id', how='all')
# ds_data

In [17]:
nb_samples = 10000
ds = xr.apply_ufunc(calc_rvs, 
                    ds_4fa[["streamflow_max_annual"]].load(), 
                    parameters.load(), 
                    parameters.scipy_dist,
                    nb_samples, 
                    input_core_dims=[['time'],['dparams'], [], []], 
                    output_core_dims=[['samples', 'time']], 
                    vectorize=True).assign_coords(samples=range(nb_samples))
param_samples = xhfa.local.fit(ds, distributions=['genextreme'])
param_samples

<xarray.Dataset>
Dimensions:                (scipy_dist: 1, samples: 10000, dparams: 3)
Coordinates: (12/17)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    time_agg               <U4 'mean'
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * scipy_dist             (scipy_dist) <U10 'genextreme'
  * samples                (samples) int32 0 1 2 3 4 ... 9996 9997 9998 9999
  * dparams                (dparams) <U5 'c' 'loc' 'scale'
Data variables:
    streamflow_max_annual  (scipy_dist, samples, dparams) float64 dask.array<chunksize=(1, 10000, 3), meta=np.ndarray>

In [18]:
# # Pour plusiuers lois
# nb_samples = 100
# ds = xr.apply_ufunc(calc_rvs, 
#                     ds_data['streamflow_max_annual'].load(), 
#                     ds_data['params'].load(), 
#                     ds_data.scipy_dist,
#                     nb_samples, 
#                     input_core_dims=[['time'],['dparams'], [], []], 
#                     output_core_dims=[['samples', 'time']], 
#                     vectorize=True).assign_coords(samples=range(nb_samples))
# ds = ds.to_dataset(name='sampled')
# ds_list = []
# for d in ds.scipy_dist.values:
#     ds_tmp = ds.sel(scipy_dist=d)
#     ds_list.append(xhfa.local.fit(ds_tmp, distributions=[d]))
# para = xr.concat(ds_list, dim='scipy_dist')
# para


In [19]:
param_samples.load()

<xarray.Dataset>
Dimensions:                (scipy_dist: 1, samples: 10000, dparams: 3)
Coordinates: (12/17)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    time_agg               <U4 'mean'
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * scipy_dist             (scipy_dist) <U10 'genextreme'
  * samples                (samples) int32 0 1 2 3 4 ... 9996 9997 9998 9999
  * dparams                (dparams) <U5 'c' 'loc' 'scale'
Data variables:
    streamflow_max_annual  (scipy_dist, samples, dparams) float64 -0.06213 .....

In [20]:
rp = xhfa.local.parametric_quantiles(param_samples, t=T)
rp
#rp.load()

<xarray.Dataset>
Dimensions:                (scipy_dist: 1, samples: 10000, return_period: 7)
Coordinates: (12/18)
    drainage_area          float32 696.0
    end_date               datetime64[ns] 2023-11-19
    id                     <U6 '023422'
    latitude               float32 46.17
    longitude              float32 -70.64
    name                   object 'Famine'
    ...                     ...
    timestep               <U1 'D'
    variable               <U10 'streamflow'
  * scipy_dist             (scipy_dist) <U10 'genextreme'
  * samples                (samples) int32 0 1 2 3 4 ... 9996 9997 9998 9999
  * return_period          (return_period) int32 20 50 100 200 1000 2000 10000
    p_quantile             (return_period) float64 0.95 0.98 ... 0.9995 0.9999
Data variables:
    streamflow_max_annual  (scipy_dist, return_period, samples) float64 290.4...

In [21]:
niv_confiance = 5
alpha_sup = (100 + niv_confiance) / 2
alpha_inf = (100 - niv_confiance) / 2

borne_inf = rp.quantile(alpha_inf/100, dim='samples')
quantile = rp.quantile(.5, dim='samples')
borne_sup = rp.quantile(alpha_sup/100, dim='samples')

In [22]:
df_boot = pd.DataFrame({"0.05": borne_inf.streamflow_max_annual.values.squeeze(),
                             "0.5": quantile.streamflow_max_annual.values.squeeze(), 
                             "0.95": borne_sup.streamflow_max_annual.values.squeeze()}, index=T)
df_boot

,0.05,0.5,0.95
20,320.972688,322.979853,325.007659
50,376.015041,379.059497,382.080654
100,418.918265,423.391187,427.693465
200,463.261119,468.919572,474.393696
1000,570.896483,581.953677,592.730667
2000,621.170407,633.310137,647.712943
10000,742.222373,762.376349,784.284090


In [23]:
df_analytic

,0.05,0.5,0.95
20,189.504826,326.978367,464.451908
50,247.170093,384.643634,522.117176
100,292.602814,430.076356,567.549897
200,339.854281,477.327822,614.801364
1000,457.403408,594.876949,732.350490
2000,511.685226,649.158767,786.632308
10000,646.996459,784.470000,921.943542
